# A.2 不在訓練資料裡的資訊怎麼使用？

# A：上下文學習與 RAG

前置：第7章prompt與對話。不需要圖片或MoE。把模型想成可以讀你遞給它的小抄的答題者：先手動給例子，再從本地文件找相關段落。檢索成功不等於回答一定正確。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像先找小抄，再核對答案來源**

找到對文件與用對資料，是兩個不同分數。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rag.svg")))

**先想一想：**問題出現沒見過日期，基模一定只能猜嗎？

想問模型訓練後才發生的資訊，可在prompt提供資料。它需要讀懂資料；如果直接把答案寫進prompt，要測的是使用資訊，不是內部記憶。

**把直覺寫成數學：**知識從外部context取得；與參數更新分開記錄。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
context = "小小書店在2027年3月1日搬到青街8號。"
prompt = f"資料：{context}\n問題：書店新地址？請只根據資料回答。"
print(prompt)
print("預期標準答案由資料確定：青街8號，模型回答仍需實際生成")

**如何讀結果：**給資料本身不是神經訓練；需要檢查引用資訊而非編造其他地址。

**只改一件事：**只換資料地址，不換問題。
